# AP3 — RAG Multimodal Agentique — version guidée et robuste
**Module :** IA Générative Multi Modale et Agentique · **Étudiante :** Salma Riad  
**Référence :** notebook de cours `rag_multimodal_agentique.ipynb` — Pr. Mohamed Youssfi.

Cette version conserve **les sept étapes** du devoir. Elle teste la présence des dépendances et des fichiers et **affiche des consignes au lieu de provoquer un arrêt brutal**. Elle ne prétend pas qu’une étape est réussie si les ressources sont absentes. **Internet est nécessaire** pour télécharger le PDF et les poids CLIP, et Ollama est nécessaire pour tester l'agent.

## 0 — Vérification de l’environnement
**Recommandation Windows :** Python 3.11 dans un environnement virtuel, puis `pip install -r requirements.txt`. Le bloc ci-dessous détecte les paquets absents et te dit exactement quoi installer. Si tu veux, passe `INSTALLER_AUTOMATIQUEMENT=True` pour lancer l'installation dans le kernel Jupyter en cours. **La première fois, l’installation peut être longue.**

In [ ]:
import sys, importlib, subprocess, pathlib, json, hashlib, urllib.request, re
from pathlib import Path
BASE = Path.cwd()
INSTALLER_AUTOMATIQUEMENT = False  # True pour tenter l'installation via le Python du notebook
paquets = {"numpy":"numpy", "fitz":"pymupdf", "torch":"torch", "open_clip":"open-clip-torch",
           "chromadb":"chromadb", "PIL":"Pillow", "langchain_core":"langchain-core",
           "langgraph":"langgraph", "langchain_ollama":"langchain-ollama"}
def modules_absents():
    return [package for module, package in paquets.items() if importlib.util.find_spec(module) is None]
missing = modules_absents()
print("Python:",sys.version.split()[0], "| Dossier:", BASE)
if missing: print("Bibliothèques absentes :", ", ".join(missing))
if missing and INSTALLER_AUTOMATIQUEMENT:
    try:
        resultat = subprocess.run([sys.executable,"-m","pip","install",*missing],check=False,timeout=900)
        importlib.invalidate_caches()
        print("Installation terminée (code retour :", resultat.returncode,"). Redémarre le kernel si demandé.")
    except Exception as e: print("Installation impossible :",type(e).__name__,str(e)[:250])
elif missing:
    print("Dans le terminal VS Code, exécute :",f'"{sys.executable}" -m pip install -r requirements.txt')
print("Le notebook ignorera proprement les étapes non disponibles.")

## 0 bis — Document source
Le TP demande le PDF **Attention Is All You Need**. Si le document n’est pas dans `pdfs/`, une tentative de téléchargement depuis arXiv est effectuée. Si elle échoue, les étapes qui nécessitent le PDF sont indiquées comme non exécutées (pas de faux contenu de remplacement).

In [ ]:
PDF_DIR=BASE/'pdfs'; PDF_DIR.mkdir(exist_ok=True)
PDF_PATH=PDF_DIR/'attention is all you need.pdf'
if not PDF_PATH.exists():
    print("PDF absent : tentative de téléchargement depuis arXiv…")
    try:
        request=urllib.request.Request('https://arxiv.org/pdf/1706.03762',headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(request,timeout=15) as response:
            data=response.read(25_000_000)
        if data[:4]==b'%PDF':
            PDF_PATH.write_bytes(data)
            print('PDF téléchargé :',PDF_PATH)
        else: print('Réponse reçue mais ce n’est pas un PDF ; aucun fichier enregistré.')
    except Exception as e:
        print('Téléchargement indisponible :',type(e).__name__)
        print('Ajoute manuellement le fichier dans :',PDF_DIR)
PDF_OK=PDF_PATH.exists() and PDF_PATH.read_bytes()[:4]==b'%PDF'
print('PDF prêt :',PDF_OK)

## Étape 1 — Extraction du texte et des images
PyMuPDF récupère les pages et les images bitmap. Pour certains dessins vectoriels, aucune image bitmap n'est détectée : c'est normal. Les images de pages rendues pourront alors servir aux essais de recherche visuelle.

In [ ]:
pages=[]; images=[]; resultat=BASE/'resultats_tp3'; resultat.mkdir(exist_ok=True)
if PDF_OK and importlib.util.find_spec('fitz'):
    try:
        import fitz
        with fitz.open(PDF_PATH) as doc:
            for page_n,page in enumerate(doc,1):
                texte=page.get_text('text',sort=True)
                pages.append({'page':page_n,'text':texte})
                for j,info in enumerate(page.get_images(full=True)):
                    try:
                        blob=doc.extract_image(info[0]); ext=blob.get('ext','png')
                        fichier=resultat/f'page_{page_n:03d}_img_{j}.{ext}'
                        fichier.write_bytes(blob['image'])
                        images.append({'page':page_n,'path':str(fichier)})
                    except Exception as e: print('Image ignorée :',type(e).__name__)
            if not images and len(doc):
                fichier=resultat/'page_001_rendu.png'
                doc[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(str(fichier))
                images.append({'page':1,'path':str(fichier)})
        (resultat/'texte.txt').write_text('\n\n'.join(p['text'] for p in pages),encoding='utf-8')
        print('Pages :',len(pages),'| Images / rendus :',len(images))
    except Exception as e: print('Extraction non validée :',type(e).__name__,str(e)[:250])
else: print('Étape 1 non exécutée : installer pymupdf et fournir le PDF.')

## Étape 2 — Embeddings texte et image avec CLIP
Le modèle ViT-B-32 place images et textes dans le même espace vectoriel et les vecteurs sont normalisés. Les poids sont téléchargés au premier usage.

In [ ]:
CLIP_OK=False; model=None; tokenizer=None; preprocess=None; device='cpu'
if pages and importlib.util.find_spec('torch') and importlib.util.find_spec('open_clip'):
    try:
        import torch, open_clip
        device='cuda' if torch.cuda.is_available() else 'cpu'
        model,_,preprocess=open_clip.create_model_and_transforms('ViT-B-32',pretrained='laion2b_s34b_b79k')
        model=model.to(device).eval(); tokenizer=open_clip.get_tokenizer('ViT-B-32')
        CLIP_OK=True; print('CLIP prêt :',device)
    except Exception as e: print('CLIP non disponible :',type(e).__name__,str(e)[:350])
else: print('CLIP non lancé : vérifie PDF, torch et open-clip-torch.')
def encode_text(text):
    if not CLIP_OK:return None
    with torch.inference_mode():
        v=model.encode_text(tokenizer([text[:260]]).to(device)).float()
        v=v/v.norm(dim=-1,keepdim=True).clamp_min(1e-12)
    return v[0].cpu().tolist()
def encode_image(path):
    if not CLIP_OK:return None
    from PIL import Image
    with Image.open(path) as im: inp=preprocess(im.convert('RGB')).unsqueeze(0).to(device)
    with torch.inference_mode():
        v=model.encode_image(inp).float(); v=v/v.norm(dim=-1,keepdim=True).clamp_min(1e-12)
    return v[0].cpu().tolist()
if CLIP_OK:print('Dimension vecteur :',len(encode_text('Transformer attention')))

## Étape 3 — Indexation dans ChromaDB
Identifiant du document basé sur SHA-256, métadonnées `type`, `page`, `source`; espaces persistants et recherche cosinus. La réindexation utilise `upsert` pour ne pas créer de doublons.

In [ ]:
INDEX_OK=False; collection=None; entries=[]
if CLIP_OK and importlib.util.find_spec('chromadb'):
    try:
        import chromadb
        doc_id=hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:16]
        client=chromadb.PersistentClient(path=str(resultat/'chroma'))
        collection=client.get_or_create_collection(name='tp3_clip_multimodal',metadata={'hnsw:space':'cosine'})
        for p in pages:
            # CLIP limite la longueur du texte : on découpe pour la recherche
            tokens=p['text'].split()
            for j in range(0,len(tokens),35):
                contenu=' '.join(tokens[j:j+35]);
                if contenu.strip():entries.append((f'{doc_id}-t-{p["page"]}-{j}',contenu,{'type':'texte','page':p['page'],'source':PDF_PATH.name}))
        for i in range(0,len(entries),16):
            batch=entries[i:i+16];vec=[encode_text(v[1]) for v in batch]
            collection.upsert(ids=[v[0] for v in batch],documents=[v[1] for v in batch],metadatas=[v[2] for v in batch],embeddings=vec)
        for j,p in enumerate(images):
            vec=encode_image(p['path'])
            collection.upsert(ids=[f'{doc_id}-im-{j}'],documents=['Image extraite ou rendu de page'],
                              metadatas=[{'type':'image','page':p['page'],'source':p['path']}],embeddings=[vec])
        INDEX_OK=collection.count()>0
        print('Index ChromaDB :',collection.count(),'éléments')
    except Exception as e:print('Indexation non validée :',type(e).__name__,str(e)[:300])
else: print('Étape 3 non exécutée : CLIP ou ChromaDB absent.')

## Étape 4 — Recherche multimodale
Recherche en cosinus par texte **ou** par image, avec les sources et la page, en renvoyant les résultats les plus proches.

In [ ]:
def rechercher(question=None,image_path=None,k=5):
    if not INDEX_OK:return []
    if (question is None)==(image_path is None):
        print('Fournir soit une question soit une image.');return []
    try:
        vecteur=encode_text(question) if question is not None else encode_image(image_path)
        r=collection.query(query_embeddings=[vecteur],n_results=min(k,collection.count()),include=['documents','metadatas','distances'])
        return [{'contenu':d,'meta':m,'similarite':round(1-float(dist),4)} for d,m,dist in zip(r['documents'][0],r['metadatas'][0],r['distances'][0])]
    except Exception as e:
        print('Recherche indisponible :',type(e).__name__,str(e)[:200]);return []
hits=rechercher(question='What is multi-head attention?',k=3)
for h in hits:print(f"p.{h['meta']['page']} ({h['meta']['type']}) score={h['similarite']} : {h['contenu'][:160]}")
if images:print('Test requête image :',len(rechercher(image_path=images[0]['path'],k=3)),'résultats')

## Étape 5 — Outil `retriever_tool`
Fonction destinée à être utilisée par un agent. Elle retourne un objet JSON contenant les extraits, leur type et la page.

In [ ]:
def retriever_tool(question:str)->str:
    hits=rechercher(question=question,k=4)
    return json.dumps([{'page':h['meta']['page'],'type':h['meta']['type'],'source':h['meta']['source'],'extrait':h['contenu'][:600]} for h in hits],ensure_ascii=False)
print('Outil défini. Résultats :',retriever_tool('encoder decoder attention')[:500])

## Étape 6 — Agent ReAct / LangGraph
Pour produire des réponses générées, un LLM local **Ollama** avec modèle `llama3.2` doit être installé et démarré. L'agent utilise l'outil de recherche et n'invente pas de références. Sans Ollama, on laisse cette étape à « non exécutée » plutôt que de simuler une réponse.

In [ ]:
AGENT_OK=False; ACTIVER_AGENT=True
if ACTIVER_AGENT and INDEX_OK and all(importlib.util.find_spec(m) for m in ('langchain_core','langgraph','langchain_ollama')):
    try:
        from langchain_core.tools import tool
        from langchain_ollama import ChatOllama
        from langgraph.prebuilt import create_react_agent
        import urllib.request
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2) as resp:
            disponibles=json.load(resp).get('models',[])
        nom=next((m.get('name') for m in disponibles if 'llama3.2' in m.get('name','')),None)
        if nom:
            @tool
            def chercher_document(question:str)->str:
                """Cherche dans le PDF des extraits sourcés avec leurs pages."""
                return retriever_tool(question)
            llm=ChatOllama(model=nom,temperature=0)
            agent=create_react_agent(llm,[chercher_document],prompt='Utilise chercher_document pour les questions sur le document; cite les pages; ne fabrique pas de sources.')
            result=agent.invoke({'messages':[('user','What is multi-head attention?')]},config={'recursion_limit':8})
            print('Réponse de l’agent :',str(result['messages'][-1].content)[:1500]); AGENT_OK=True
        else:print('Agent non testé : exécuter « ollama pull llama3.2 » et démarrer Ollama.')
    except Exception as e:print('Agent non exécuté :',type(e).__name__,str(e)[:220])
else:print('Agent non testé : index, dépendances ou option ACTIVER_AGENT manquante.')

## Étape 7 — Bilan et validation
Les validations indiquent **OK** ou **NON VALIDÉ**. Seules les vérifications réellement exécutées peuvent être déclarées réussies.

In [ ]:
tests={'pdf_valide':PDF_OK,'extraction':bool(pages),'images':bool(images),'embeddings_clip':CLIP_OK,
       'indexation':INDEX_OK,'retrieval_texte':bool(hits),'outil_agent':bool(retriever_tool('attention')),
       'agent_langgraph_execute':AGENT_OK}
for nom, ok in tests.items():print(f'{nom:28} :', 'OK' if ok else 'NON VALIDÉ')
(resultat/'rapport_validation.json').write_text(json.dumps(tests,indent=2,ensure_ascii=False),encoding='utf-8')
print('Rapport :',resultat/'rapport_validation.json')

### Important pour le rendu
Ce notebook est prêt à être **testé**, mais une exécution réussie de toutes les fonctionnalités demande les paquets Python, le PDF, les poids CLIP et Ollama. Si le rapport contient `NON VALIDÉ`, l'étape correspondante n'est **pas terminée**. Avant GitHub, vérifier le rapport, l'indexation, les citations documentaires et le comportement de l'agent.